In [ ]:
import cv2
import numpy as np
import os
from Expirements.SceneSegmentation.utils import calculate_interval_metric

In [ ]:
def detectShots(video_path,
                min_len=10,
                history_size=500,
                k=2.5):

    os.makedirs("debug/shots", exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    prev_hsv = None
    prev_edges = None

    history = []

    cuts = [0]
    fi = 0
    current_length = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame, (160, 90))

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 100, 200)

        current_length += 1

        if prev_hsv is not None:

            hsv_diff = np.abs(
                hsv.astype(np.int32) -
                prev_hsv.astype(np.int32)
            ).mean()

            edge_diff = np.mean(edges != prev_edges)

            score = hsv_diff + 15 * edge_diff

            if len(history) >= 10:

                mu = np.mean(history)
                sigma = np.std(history)

                adaptive_threshold = mu + k * sigma

                if score > adaptive_threshold:

                    cuts.append(fi)

                    cv2.imwrite(
                        f"debug/shots/shot_{len(cuts):04d}_frame_{fi}.jpg",
                        frame
                    )

                    current_length = 0
                    history = history[-10:]

            history.append(score)

            if len(history) > history_size:
                history.pop(0)

        prev_hsv = hsv
        prev_edges = edges

        fi += 1

    cap.release()

    # always add the last frame as a cut
    cuts.append(fi)

    #clean cuts that are too close together
    cleaned_cuts = [cuts[0]]
    for i in range(1, len(cuts)):
        start = cuts[i-1]
        end = cuts[i]
        if (end - start) < min_len:
            cleaned_cuts.append(end)

    return cleaned_cuts, fi, fps

In [ ]:
video_paths = [
    "./dataset/videos/La_Chute_dune_Plume.mp4",
    "./dataset/videos/tears_of_steel_1080p.mp4",
    "./dataset/videos/sintel-1024-surround.mp4",
]
gt_files = [
    "./dataset/videos/La_Chute_dune_Plume_scenes.txt",
    "./dataset/videos/tears_of_steel_1080p_scenes.txt",
    "./dataset/videos/sintel-1024-surround_scenes.txt",
]

In [ ]:
for video_path, gt_file in zip(video_paths, gt_files):
    print(f"video: {video_path}")

    cuts, total_frames, fps = detectShots(video_path)

    pred_intervals = [(cuts[i], cuts[i+1] - 1) for i in range(len(cuts) - 1)]

    gt_intervals = []
    with open(gt_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                gt_intervals.append((int(parts[0]), int(parts[1])))

    print(f"  shots detected: {len(pred_intervals)}  gt scenes: {len(gt_intervals)}")
    for m in ['precision', 'recall', 'f1', 'iou']:
        print(f"  {m}: {calculate_interval_metric([gt_intervals], [pred_intervals], m):.3f}")
    print()